In [ ]:
import tensorflow as tf
import kagglehub
import matplotlib.pyplot as plt

dataset_path = kagglehub.dataset_download("odins0n/handm-dataset-128x128")



100%|██████████| 1.19G/1.19G [00:34<00:00, 37.0MB/s]


Extracting files...


In [ ]:
print("Loading the images and spliting into train/ validation sets...")

Loading the images and spliting into train/ validation sets...


In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split = 0.2,
    subset = "training",
    seed=123,
    image_size=(224,224),
    batch_size=32
)


Found 105100 files belonging to 1 classes.
Using 84080 files for training.


In [ ]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224,224),
    batch_size=32
)

Found 105100 files belonging to 1 classes.
Using 21020 files for validation.


In [ ]:
class_names = train_ds.class_names
print("Class_Names : ", class_names)

Class_Names :  ['images_128_128']


In [ ]:
for images, labels in train_ds.take(1):
    print(images.shape)

(32, 224, 224, 3)


In [ ]:
# setup the transform learning

base_model = tf.keras.applications.MobileNetV2(
    input_shape=[224,224,3],
    include_top=False,
    weights='imagenet'
)




In [ ]:
#freeze the base model

base_model.trainable = False

In [ ]:
# build the new model 
inputs = tf.keras.Input(shape=(224,224,3))

In [ ]:
#scale the pixels to what MobileNet expects(-1 to 1)

x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)

x = base_model(x, training=False)

# flatten the features
x = tf.keras.layers.GlobalAveragePooling2D()(x)

#the final output layer: % units
outputs =  tf.keras.layers.Dense(5, activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

In [ ]:
# complie the model

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data = val_ds,
    epochs=1,
)

 707/2628 ━━━━━━━━━━━━━━━━━━━━ 44:46 1s/step - accuracy: 0.9936 - loss: 0.0259

In [ ]:
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.plot(history.history["accuracy"], label = "Train Accuracy")
plt.plot(history.history["val_accuracy"], label = "val Accuracy")
plt.legend()

plt.subplot(1,2,1)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="val_loss")
plt.legend()
plt.show()